# JAX analog of `vmap` + `torch.compile(backend='inductor')` correctness bug check

The torch notebook (`vmap_inductor_bug_reproducer.ipynb`) shows that for Z2-symmetric fermionic PEPS with **odd block dimensions >= 5** (D=10,14,18,...), `torch.compile` after `torch.export` + `torch.vmap` produces **wrong results**.

This notebook checks whether the same bug exists in JAX, i.e. whether `jax.jit(jax.vmap(amplitude))` disagrees with eager (non-jitted) evaluation.

**Setup:** exact contraction (no SVD/QR truncation), same as the torch reproducer.

In [1]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
# set jax dtype to 64
os.environ.setdefault("JAX_ENABLE_X64", "1")

import numpy as np
import jax
import jax.numpy as jnp
import quimb.tensor as qtn
import symmray as sr

print(f"JAX version: {jax.__version__}")
print(f"JAX devices: {jax.devices()}")
print(f"JAX default backend: {jax.default_backend()}")
jax.config.update('jax_enable_x64', True) 

JAX version: 0.8.0
JAX devices: [CudaDevice(id=0)]
JAX default backend: gpu


In [2]:
Lx, Ly = 6, 4
nsites = Lx * Ly
B = 2  # batch size
N_f = nsites - 2  # number of fermions

print(f"System: {Lx}x{Ly} lattice, {nsites} sites, {N_f} fermions, B={B}")

System: 6x4 lattice, 24 sites, 22 fermions, B=2


## Define amplitude function

In [3]:
def amplitude(x, params, skeleton):
    """Single-sample exact contraction of fermionic PEPS."""
    tn = qtn.unpack(params, skeleton)
    tnx = tn.isel(
        {tn.site_ind(site): x[i] for i, site in enumerate(tn.sites)}
    )
    # exact contraction — no SVD/QR truncation
    return tnx.contract()

## Generate random configs

Using numpy RNG (different seeds than torch notebook, but exact config values don't matter for the bug test — we just need valid fermionic configs).

In [4]:
def random_initial_config(N_f, N_sites, seed):
    """Generate a random fermionic config."""
    rng = np.random.default_rng(seed)
    config = np.array([1, 2] * (N_sites // 2), dtype=np.int32)
    n_empty = N_sites - N_f
    config[:n_empty] = 0
    config = rng.permutation(config)
    assert np.sum(config == 1) == N_f // 2
    assert np.sum(config == 2) == N_f // 2
    return config

xs_np = np.stack([
    random_initial_config(N_f, nsites, seed=s) for s in range(B)
])
xs = jnp.array(xs_np)
print(f"Configs shape: {xs.shape}")
xs

Configs shape: (2, 24)


Array([[1, 1, 2, 1, 2, 1, 1, 1, 2, 2, 1, 1, 0, 1, 1, 2, 2, 2, 2, 2, 1, 2,
        0, 2],
       [0, 2, 2, 2, 1, 2, 1, 2, 1, 2, 1, 1, 1, 2, 1, 2, 1, 0, 2, 1, 1, 2,
        1, 2]], dtype=int32)

## Test function: compare eager vs jax.jit(jax.vmap(...))

In [5]:
def test_D(D):
    """Test eager vs jax.jit(jax.vmap(...)) for a given bond dim D."""
    print(f"\n{'=' * 60}")
    print(f"D={D}, Z2 block dim={D // 2}, exact contraction, {Lx}x{Ly}")
    print(f"{'=' * 60}")

    # Random Z2-symmetric fermionic PEPS (same params as torch notebook)
    peps = sr.networks.PEPS_fermionic_rand(
        "Z2", Lx, Ly, D,
        phys_dim=[
            (0, 0),
            (1, 0),
            (1, 1),
            (0, 1),
        ],
        subsizes="equal",
        flat=True,
        seed=42,
        dtype="float64",
    )

    # Pack into pytree params + skeleton
    params, skeleton = qtn.pack(peps)

    # --- Eager baseline (no jit, loop over samples) ---
    eager_results = []
    for i in range(B):
        val = amplitude(xs[i], params, skeleton)
        eager_results.append(float(val))
    eager_results = jnp.array(eager_results)
    print(f"  Eager (loop):  {eager_results}")

    # --- jax.jit(jax.vmap(...)) ---
    def amp_for_vmap(x, params):
        return amplitude(x, params, skeleton)

    amp_vmap_jit = jax.jit(jax.vmap(amp_for_vmap, in_axes=(0, None)))

    # Warmup + compute
    vmap_results = amp_vmap_jit(xs, params)
    vmap_results = jax.block_until_ready(vmap_results)
    print(f"  vmap+jit:      {vmap_results}")

    # Compare
    diff = float(jnp.max(jnp.abs(eager_results - vmap_results)))
    scale = float(jnp.max(jnp.abs(eager_results)))
    rel_diff = diff / scale if scale > 0 else diff
    ok = rel_diff < 1e-6
    print(f"  Rel diff: {rel_diff:.2e}  {'PASS' if ok else 'FAIL'}")
    print(f"    eager:    {eager_results}")
    print(f"    vmap+jit: {vmap_results}")

    return ok, rel_diff

## Run tests across bond dimensions

In [6]:
D_list = [8, 10, 12, 14, 16, 18, 20]
results = {}
for D in D_list:
    results[D] = test_D(D)


D=8, Z2 block dim=4, exact contraction, 6x4


/home/sijingdu/.local/share/uv/python/cpython-3.12.9-linux-x86_64-gnu/lib/python3.12/importlib/__init__.py:90: FutureWarning: The module 'quimb.tensor.tensor_arbgeom' is deprecated and will be removed in a future release. Most functionality can be still be accessed directly from 'quimb.tensor' instead. The actual implementations have moved to `quimb.tensor.tnag.core`.
  return _bootstrap._gcd_import(name[level:], package, level)


  Eager (loop):  [ 1.16143769e+13 -1.08326176e+14]
  vmap+jit:      [ 1.16143769e+13 -1.08326176e+14]
  Rel diff: 6.31e-16  PASS
    eager:    [ 1.16143769e+13 -1.08326176e+14]
    vmap+jit: [ 1.16143769e+13 -1.08326176e+14]

D=10, Z2 block dim=5, exact contraction, 6x4
  Eager (loop):  [ 1.85556217e+15 -1.72752362e+15]
  vmap+jit:      [ 1.85556217e+15 -1.72752362e+15]
  Rel diff: 9.43e-16  PASS
    eager:    [ 1.85556217e+15 -1.72752362e+15]
    vmap+jit: [ 1.85556217e+15 -1.72752362e+15]

D=12, Z2 block dim=6, exact contraction, 6x4
  Eager (loop):  [-1.59372158e+17 -6.41908432e+16]
  vmap+jit:      [-1.59372158e+17 -6.41908432e+16]
  Rel diff: 1.56e-15  PASS
    eager:    [-1.59372158e+17 -6.41908432e+16]
    vmap+jit: [-1.59372158e+17 -6.41908432e+16]

D=14, Z2 block dim=7, exact contraction, 6x4
  Eager (loop):  [4.60721770e+17 2.92861133e+17]
  vmap+jit:      [4.60721770e+17 2.92861133e+17]
  Rel diff: 8.47e-15  PASS
    eager:    [4.60721770e+17 2.92861133e+17]
    vmap+jit: [4

## Summary

In [7]:
print(f"\n{'=' * 60}")
print("JAX: Compare eager vs jax.jit(jax.vmap(amplitude))")
print(f"({Lx}x{Ly} lattice, exact contraction, XLA backend)")
print(f"{'=' * 60}")
print(
    f"  {'D':>4}  {'Z2 block dim':>12}"
    f"  {'eager==vmap+jit?':>16}  {'max rel diff':>12}"
)
print(f"  {'-'*4}  {'-'*12}  {'-'*16}  {'-'*12}")
for D in D_list:
    ok, diff = results[D]
    bdim = D // 2
    status = "PASS" if ok else "FAIL"
    print(f"  {D:>4}  {bdim:>12}  {status:>16}  {diff:>12.2e}")
print(f"{'=' * 60}")

print("\nFor reference, torch inductor bug pattern:")
print("  D=8(4):PASS, D=10(5):FAIL, D=12(6):PASS, D=14(7):FAIL,")
print("  D=16(8):PASS, D=18(9):FAIL, D=20(10):PASS")


JAX: Compare eager vs jax.jit(jax.vmap(amplitude))
(6x4 lattice, exact contraction, XLA backend)
     D  Z2 block dim  eager==vmap+jit?  max rel diff
  ----  ------------  ----------------  ------------
     8             4              PASS      6.31e-16
    10             5              PASS      9.43e-16
    12             6              PASS      1.56e-15
    14             7              PASS      8.47e-15
    16             8              PASS      4.71e-15
    18             9              PASS      3.81e-15
    20            10              PASS      3.45e-15

For reference, torch inductor bug pattern:
  D=8(4):PASS, D=10(5):FAIL, D=12(6):PASS, D=14(7):FAIL,
  D=16(8):PASS, D=18(9):FAIL, D=20(10):PASS
